# CLV Prediction — Assignment 1
**Goal:** Predict 2018-2019 revenue for customers of a European fashion shoe shop using 2016-2017 transaction data.  
**Evaluation:** MAE (primary), Spearman correlation (secondary). Benchmark to beat: MAE 61.15 / Spearman 0.41.

## Notebook structure
1. Setup & Data Loading  
2. Feature Engineering  
3. Exploratory Analysis & Preprocessing  
4. Modeling — Approach 1: Two-stage pipeline (Classifier + Regressor)  
5. Modeling — Approach 2: Direct Regression  
6. Modeling — Approach 3: LGBM optimizing MAE *(best)*  
7. Alternative Algorithms  
8. Automated Feature Engineering (FeatureTools)  
9. Probabilistic Models (BG/NBD + Gamma-Gamma)  
10. Error Analysis  
11. Clustering Experiments  
12. Final Model & Results Summary

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import copy
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import Ridge, Lasso
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr

# Load clean transaction data (pre-processed by teammate in data_clean_split.ipynb)
df_train = pd.read_csv("data/train_clean.csv", parse_dates=["order_date", "pack_date"])
df_valid = pd.read_csv("data/valid_clean.csv", parse_dates=["order_date", "pack_date"])

df_train_targets = pd.read_csv("data/train_targets.csv")
df_valid_targets = pd.read_csv("data/valid_targets.csv")

print("Train transactions:", df_train.shape)
print("Valid transactions:", df_valid.shape)
print("Train targets:     ", df_train_targets.shape)
print("Valid targets:     ", df_valid_targets.shape)

Train transactions: (219128, 27)
Valid transactions: (55183, 27)
Train targets:      (93272, 2)
Valid targets:      (23319, 2)


In [2]:
# Reference date: last day of the transaction period
# Used to calculate recency (how many days since last purchase)
reference_date = pd.Timestamp("2017-12-31")

## 2. Feature Engineering
All features are aggregated at customer level (one row per customer).  
We build three feature blocks and merge them:
- **RFM + behavior features** — recency, frequency, monetary, returns, discounts, delivery, trend
- **Seasonality features** — when during the year the customer buys
- **Product profile features** — what types of products the customer buys

In [3]:
def compute_rfm_features(df):
    """
    Computes RFM and purchase behavior features, aggregated at customer level.
    Input:  transaction-level dataframe (one row per item)
    Output: one row per customer
    """
    df = df.copy()
    # Avoid categorical dtype issues that can cause memory errors
    df["cust_id"] = df["cust_id"].astype(str)
    df["sale_id"]  = df["sale_id"].astype(str)

    # Separate returned and non-returned items
    df_net = df[df["is_returned"] == 0]

    # ── Recency: days since last purchase ──────────────────────────────────
    recency = (
        df.groupby("cust_id")["order_date"]
        .max()
        .apply(lambda x: (reference_date - x).days)
        .rename("recency_days")
    )

    # ── Frequency: unique orders with at least 1 non-returned item ─────────
    frequency = (
        df_net.groupby("cust_id")["sale_id"]
        .nunique()
        .rename("frequency")
    )

    # ── One-time buyer flag ─────────────────────────────────────────────────
    is_one_time_buyer = (frequency == 1).astype(int).rename("is_one_time_buyer")

    # ── Days between orders (only meaningful for repeat buyers) ────────────
    order_dates_per_cust = (
        df.drop_duplicates(subset=["cust_id", "sale_id"])
        .sort_values(["cust_id", "order_date"])
        .groupby("cust_id")["order_date"]
    )
    avg_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.mean() if len(x) > 1 else np.nan)
        .rename("avg_days_between_orders")
    )
    std_days_between_orders = (
        order_dates_per_cust
        .apply(lambda x: x.diff().dt.days.std() if len(x) > 1 else np.nan)
        .rename("std_days_between_orders")
    )

    # ── Monetary: total revenue excluding returns ───────────────────────────
    total_revenue = (
        df_net.groupby("cust_id")["sale_revenue"]
        .sum()
        .rename("total_revenue")
    )

    # ── Average ticket: revenue per effective order ─────────────────────────
    avg_ticket = (total_revenue / frequency).rename("avg_ticket")

    # ── Items per order (gross): includes returns ───────────────────────────
    items_per_order_gross = (
        df.groupby(["cust_id", "sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_gross")
    )

    # ── Items per order (net): excludes returns ─────────────────────────────
    items_per_order_net = (
        df_net.groupby(["cust_id", "sale_id"])
        .size()
        .groupby("cust_id").mean()
        .rename("items_per_order_net")
    )

    # ── Return behavior ─────────────────────────────────────────────────────
    total_returns = df.groupby("cust_id")["is_returned"].sum().rename("total_returns")
    total_items   = df.groupby("cust_id")["is_returned"].count().rename("total_items")
    return_rate   = (total_returns / total_items).rename("return_rate")

    # ── Delivery time ───────────────────────────────────────────────────────
    df["delivery_days"] = (df["pack_date"] - df["order_date"]).dt.days
    avg_delivery = (
        df.groupby("cust_id")["delivery_days"]
        .mean()
        .rename("avg_delivery_days")
    )

    # ── Discount behavior ───────────────────────────────────────────────────
    df["has_discount_item"] = (df["sale_discount_applied"] < 0).astype(int)
    total_discounted_items = (
        df.groupby("cust_id")["has_discount_item"].sum().rename("total_discounted_items")
    )
    discount_rate = (
        df.groupby("cust_id")["has_discount_item"].mean().rename("discount_rate")
    )

    # ── Trend features: is the customer becoming more active over time? ─────
    df["year"] = df["order_date"].dt.year
    orders_2017  = df[df["year"] == 2017].groupby("cust_id").size()
    orders_total = df.groupby("cust_id").size()
    pct_orders_2017 = (orders_2017 / orders_total).fillna(0).rename("pct_orders_2017")

    revenue_2016 = (
        df[df["year"] == 2016].groupby("cust_id")["sale_revenue"]
        .mean().rename("avg_revenue_2016")
    )
    revenue_2017 = (
        df[df["year"] == 2017].groupby("cust_id")["sale_revenue"]
        .mean().rename("avg_revenue_2017")
    )
    revenue_trend = (revenue_2017 - revenue_2016).rename("revenue_trend")

    rfm = pd.concat([
        recency, frequency, total_revenue, avg_ticket,
        items_per_order_gross, items_per_order_net,
        total_returns, total_items, return_rate,
        total_discounted_items, discount_rate,
        avg_days_between_orders, std_days_between_orders,
        is_one_time_buyer, pct_orders_2017, revenue_trend
    ], axis=1).reset_index()

    return rfm

In [4]:
def compute_seasonality_features(df):
    """
    Captures when during the year the customer tends to buy.
    We focus on January and July (peak sales months).
    most_active_month was discarded due to too many ties.
    """
    df = df.copy()
    df["order_month"] = df["order_date"].dt.month

    buys_in_jan = (
        df.groupby("cust_id")["order_month"]
        .apply(lambda x: int((x == 1).any()))
        .rename("buys_in_jan")
    )
    buys_in_jul = (
        df.groupby("cust_id")["order_month"]
        .apply(lambda x: int((x == 7).any()))
        .rename("buys_in_jul")
    )
    return pd.concat([buys_in_jan, buys_in_jul], axis=1).reset_index()

In [5]:
def compute_product_features(df):
    """
    Captures diversity of purchasing behavior across product categories,
    brands, and customer segments.
    """
    df = df.copy()

    n_genders = df.groupby("cust_id")["prod_type_1"].nunique().rename("n_genders")
    buys_women = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "women").any())).rename("buys_women")
    )
    buys_men = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int((x == "men").any())).rename("buys_men")
    )
    buys_kids = (
        df.groupby("cust_id")["prod_type_1"]
        .apply(lambda x: int(x.isin(["boys", "girls"]).any())).rename("buys_kids")
    )
    n_product_categories = (
        df.groupby("cust_id")["prod_type_3"].nunique().rename("n_product_categories")
    )
    n_brands = (
        df.groupby("cust_id")["prod_brand"].nunique().rename("n_brands")
    )
    pct_web_only = (
        df.groupby("cust_id")["prod_web_only"].mean().rename("pct_web_only")
    )
    buys_outlet = (
        df.groupby("cust_id")["prod_outlet"]
        .apply(lambda x: int((x > 0).any())).rename("buys_outlet")
    )
    return pd.concat([
        n_genders, buys_women, buys_men, buys_kids,
        n_product_categories, n_brands, pct_web_only, buys_outlet
    ], axis=1).reset_index()

In [6]:
def build_features(df):
    """Combines all feature blocks into one customer-level table."""
    rfm         = compute_rfm_features(df)
    seasonality = compute_seasonality_features(df)
    product     = compute_product_features(df)
    return rfm.merge(seasonality, on="cust_id").merge(product, on="cust_id")

df_train_features = build_features(df_train)
df_valid_features = build_features(df_valid)

print("Train features shape:", df_train_features.shape)
print("Valid features shape:", df_valid_features.shape)

Train features shape: (93272, 27)
Valid features shape: (23319, 27)


In [7]:
# Merge features with targets
df_train_final = df_train_features.merge(df_train_targets, on="cust_id")
df_valid_final = df_valid_features.merge(df_valid_targets, on="cust_id")

print("Train final shape:", df_train_final.shape)
print("Valid final shape:", df_valid_final.shape)

Train final shape: (93272, 28)
Valid final shape: (23319, 28)


## 3. Exploratory Analysis & Preprocessing

In [8]:
target = df_train_final["revenue_2018_2019"]
print("==== Target distribution ====")
print(f"Customers with revenue = 0: {(target==0).sum()} ({100*(target==0).mean():.1f}%)")
print(f"Customers with revenue > 0: {(target>0).sum()} ({100*(target>0).mean():.1f}%)")
print(f"\nMean (all):     {target.mean():.2f}")
print(f"Mean (>0 only): {target[target>0].mean():.2f}")
print(f"Max:            {target.max():.2f}")

print("\n=== NaNs per column ===")
nans = df_train_final.isna().sum()
print(nans[nans > 0])

==== Target distribution ====
Customers with revenue = 0: 59196 (63.5%)
Customers with revenue > 0: 34076 (36.5%)

Mean (all):     70.18
Mean (>0 only): 192.08
Max:            1197.94

=== NaNs per column ===
frequency                      7
total_revenue                  7
avg_ticket                     7
items_per_order_net            7
avg_days_between_orders    61971
std_days_between_orders    78984
is_one_time_buyer              7
revenue_trend              77146
dtype: int64


In [9]:
# ── NaN handling ─────────────────────────────────────────────────────────────
# Customers who returned everything → fill with 0
cols_fill_zero = ["frequency", "total_revenue", "avg_ticket",
                  "items_per_order_net", "is_one_time_buyer"]
df_train_final[cols_fill_zero] = df_train_final[cols_fill_zero].fillna(0)
df_valid_final[cols_fill_zero] = df_valid_final[cols_fill_zero].fillna(0)

# revenue_trend NaN = customer only bought in one year → no trend → fill with 0
df_train_final["revenue_trend"] = df_train_final["revenue_trend"].fillna(0)
df_valid_final["revenue_trend"] = df_valid_final["revenue_trend"].fillna(0)

# One-time buyers → fill days between orders with train median
for col in ["avg_days_between_orders", "std_days_between_orders"]:
    median_val = df_train_final[col].median()
    df_train_final[col] = df_train_final[col].fillna(median_val)
    df_valid_final[col] = df_valid_final[col].fillna(median_val)

print("Remaining NaNs in train:", df_train_final.isna().sum().sum())
print("Remaining NaNs in valid:", df_valid_final.isna().sum().sum())

Remaining NaNs in train: 0
Remaining NaNs in valid: 0


In [10]:
# ── VIP feature ──────────────────────────────────────────────────────────────
# Identified through error analysis: high-value customers are systematically underpredicted
# Thresholds based on train distribution: frequency >= 4 (top quartile), revenue >= 249 (top 10%)
# NOTE: thresholds derived from train only → no data leaking
df_train_final["is_vip"] = (
    (df_train_final["frequency"] >= 4) &
    (df_train_final["total_revenue"] >= 249)
).astype(int)
df_valid_final["is_vip"] = (
    (df_valid_final["frequency"] >= 4) &
    (df_valid_final["total_revenue"] >= 249)
).astype(int)

print("VIP customers in train:", df_train_final["is_vip"].sum())
print("VIP customers in valid:", df_valid_final["is_vip"].sum())

# ── Define feature matrix ─────────────────────────────────────────────────────
feature_cols = [col for col in df_train_final.columns
                if col not in ["cust_id", "revenue_2018_2019"]]

X_train = df_train_final[feature_cols]
y_train = df_train_final["revenue_2018_2019"]
X_valid = df_valid_final[feature_cols]
y_valid = df_valid_final["revenue_2018_2019"]

y_train_binary = (y_train > 0).astype(int)
y_valid_binary = (y_valid > 0).astype(int)

print("\nFeature count:", len(feature_cols))
print("X_train shape:", X_train.shape)

VIP customers in train: 4677
VIP customers in valid: 1202

Feature count: 27
X_train shape: (93272, 27)


## 4. Approach 1 — Two-Stage Pipeline (Classifier + Regressor)
**Motivation:** 63.5% of customers have zero revenue — first classify who returns, then predict how much.  
**Result:** MAE ~71. Worse than direct regression. Kept for comparison.

In [11]:
# ── Stage 1: RandomForest Classifier ─────────────────────────────────────────
# Best params from grid search: max_depth=6, min_samples_leaf=20, n_estimators=200
# Grid search (commented out):
# param_grid = {"n_estimators": [100,200,300], "max_depth": [6,8,10], "min_samples_leaf": [10,20,30]}
# grid_search = GridSearchCV(RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
#                            param_grid, cv=5, scoring="f1", verbose=1, n_jobs=-1)
# grid_search.fit(X_train, y_train_binary)  # → max_depth=6, min_samples_leaf=20, n_estimators=200

clf_best = RandomForestClassifier(
    max_depth=6, min_samples_leaf=20, n_estimators=200,
    class_weight="balanced", random_state=42, n_jobs=-1
)
clf_best.fit(X_train, y_train_binary)
print(classification_report(y_valid_binary, clf_best.predict(X_valid)))

              precision    recall  f1-score   support

           0       0.75      0.77      0.76     14737
           1       0.59      0.56      0.57      8582

    accuracy                           0.69     23319
   macro avg       0.67      0.67      0.67     23319
weighted avg       0.69      0.69      0.69     23319



In [12]:
# ── Stage 2: XGBoost Regressor on returners only ─────────────────────────────
# Best params from grid search: learning_rate=0.01, max_depth=4, n_estimators=500, subsample=0.8
# Grid search (commented out):
# param_grid_xgb = {"n_estimators": [200,300,500], "max_depth": [4,6,8],
#                   "learning_rate": [0.01,0.05,0.1], "subsample": [0.7,0.8]}
# grid_search_xgb = GridSearchCV(XGBRegressor(...), param_grid_xgb, cv=5,
#                                scoring="neg_mean_absolute_error", n_jobs=-1)
# grid_search_xgb.fit(X_train_reg, y_train_reg_log)
# → learning_rate=0.01, max_depth=4, n_estimators=500, subsample=0.8

train_returners = df_train_final[df_train_final["revenue_2018_2019"] > 0]
X_train_reg     = train_returners[feature_cols]
y_train_reg_log = np.log1p(train_returners["revenue_2018_2019"])

clf_xgb = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train_binary==0).sum() / (y_train_binary==1).sum(),
    random_state=42, n_jobs=-1, verbosity=0
)
clf_xgb.fit(X_train, y_train_binary)

reg_xgb_2stage = XGBRegressor(
    n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.8,
    colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0
)
reg_xgb_2stage.fit(X_train_reg, y_train_reg_log)

# Full pipeline
y_pred_proba  = clf_xgb.predict_proba(X_valid)[:, 1]
returner_mask = (y_pred_proba >= 0.5)
y_pred_2stage = np.zeros(len(X_valid))
y_pred_2stage[returner_mask] = np.expm1(reg_xgb_2stage.predict(X_valid[returner_mask]))

print(f"MAE (two-stage):      {mean_absolute_error(y_valid, y_pred_2stage):.2f}")
print(f"Spearman (two-stage): {spearmanr(y_valid, y_pred_2stage).statistic:.3f}")

MAE (two-stage):      70.80
Spearman (two-stage): 0.404


## 5. Approach 2 — Direct XGBoost Regression
**Key insight:** Train on ALL customers including zeros. XGBoost learns to predict 0 for non-returners.  
**Why log transform?** Revenue is right-skewed. log1p compresses large values, reversed with expm1.  
**Result:** MAE 66.01 — significant improvement over two-stage.

In [13]:
# Best params from grid search: learning_rate=0.05, max_depth=4, n_estimators=700, subsample=0.7
# Grid search (commented out):
# param_grid_direct = {"n_estimators": [300,500,700], "max_depth": [3,4,6],
#                      "learning_rate": [0.01,0.05], "subsample": [0.7,0.8]}
# grid_search_direct = GridSearchCV(XGBRegressor(colsample_bytree=0.8, random_state=42,
#                                   n_jobs=-1, verbosity=0), param_grid_direct, cv=5,
#                                   scoring="neg_mean_absolute_error", verbose=1, n_jobs=-1)
# grid_search_direct.fit(X_train, np.log1p(y_train))
# → learning_rate=0.05, max_depth=4, n_estimators=700, subsample=0.7

reg_xgb_direct = XGBRegressor(
    n_estimators=700, max_depth=4, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
reg_xgb_direct.fit(X_train, np.log1p(y_train))

y_pred_direct = np.expm1(reg_xgb_direct.predict(X_valid))
y_pred_direct = np.clip(y_pred_direct, 0, None)

print(f"MAE (XGBoost direct):      {mean_absolute_error(y_valid, y_pred_direct):.2f}")
print(f"Spearman (XGBoost direct): {spearmanr(y_valid, y_pred_direct).statistic:.3f}")

MAE (XGBoost direct):      65.92
Spearman (XGBoost direct): 0.391


## 6. Approach 3 — LGBM optimizing MAE *(best approach)*
**Key insight:** Optimizing MAE directly instead of MSE (default) focuses the model on 
minimizing absolute errors rather than penalizing large outliers more.  
**Result:** MAE 63.05 — best model.

In [14]:
# ── LGBM with MAE objective ──────────────────────────────────────────────────
reg_lgbm_mae = LGBMRegressor(
    objective="mae",
    n_estimators=700, max_depth=4, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
reg_lgbm_mae.fit(X_train, np.log1p(y_train))

y_pred_lgbm_mae = np.expm1(reg_lgbm_mae.predict(X_valid))
y_pred_lgbm_mae = np.clip(y_pred_lgbm_mae, 0, None)

print(f"MAE (LGBM MAE):      {mean_absolute_error(y_valid, y_pred_lgbm_mae):.2f}")
print(f"Spearman (LGBM MAE): {spearmanr(y_valid, y_pred_lgbm_mae).statistic:.3f}")

MAE (LGBM MAE):      64.02
Spearman (LGBM MAE): 0.385


In [15]:
# ── Grid search over LGBM hyperparameters ────────────────────────────────────
# Best params: learning_rate=0.01, max_depth=6, n_estimators=1000, subsample=0.7
# Grid search (commented out):
# param_grid_lgbm = {"n_estimators": [500,700,1000], "max_depth": [4,6],
#                    "learning_rate": [0.01,0.05], "subsample": [0.7,0.8]}
# grid_search_lgbm = GridSearchCV(LGBMRegressor(objective="mae", colsample_bytree=0.8,
#                                 random_state=42, n_jobs=-1, verbose=-1),
#                                 param_grid_lgbm, cv=5,
#                                 scoring="neg_mean_absolute_error", verbose=1, n_jobs=-1)
# grid_search_lgbm.fit(X_train, np.log1p(y_train))
# → learning_rate=0.01, max_depth=6, n_estimators=1000, subsample=0.7

reg_lgbm_best = LGBMRegressor(
    objective="mae",
    learning_rate=0.01, max_depth=6, n_estimators=1000,
    subsample=0.7, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
reg_lgbm_best.fit(X_train, np.log1p(y_train))

y_pred_lgbm_best = np.expm1(reg_lgbm_best.predict(X_valid))
y_pred_lgbm_best = np.clip(y_pred_lgbm_best, 0, None)

print(f"MAE (LGBM MAE grid search):      {mean_absolute_error(y_valid, y_pred_lgbm_best):.2f}")
print(f"Spearman (LGBM MAE grid search): {spearmanr(y_valid, y_pred_lgbm_best).statistic:.3f}")

MAE (LGBM MAE grid search):      63.13
Spearman (LGBM MAE grid search): 0.391


In [16]:
# ── Feature importance ────────────────────────────────────────────────────────
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": reg_lgbm_best.feature_importances_
}).sort_values("importance", ascending=False)

print(feature_importance.to_string(index=False))

                feature  importance
           recency_days        4603
avg_days_between_orders        2412
        pct_orders_2017        2338
   n_product_categories        2025
          total_revenue        2012
              buys_kids        1939
  items_per_order_gross        1770
               n_brands        1577
            total_items        1474
             avg_ticket        1415
               buys_men        1203
              n_genders        1003
              frequency         928
          discount_rate         923
 total_discounted_items         705
std_days_between_orders         635
             buys_women         604
      is_one_time_buyer         482
            return_rate         461
    items_per_order_net         349
          revenue_trend         321
           pct_web_only         316
          total_returns         241
            buys_in_jul          98
            buys_in_jan          92
                 is_vip          72
            buys_outlet     

In [17]:
# ── Feature selection: keep features with importance >= 200 ──────────────────
# buys_outlet has importance=0, n_genders and is_vip have very low importance
# Removing low-importance features reduces noise
top_features = feature_importance[feature_importance["importance"] >= 200]["feature"].tolist()
print(f"Features with importance >= 200: {len(top_features)}")

X_train_top = df_train_final[top_features]
X_valid_top = df_valid_final[top_features]

Features with importance >= 200: 23


In [18]:
# ── Manual candidate search with num_leaves and regularization ───────────────
# Exploring LightGBM-specific parameters not covered in GridSearchCV
# Best result: Trial 3 → MAE 63.05 | Spearman 0.391
# num_leaves=31, min_child_samples=20, learning_rate=0.005, n_estimators=5000
# subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0
#
# candidate_params = [
#     {"num_leaves": 15, "min_child_samples": 20, "learning_rate": 0.05, "n_estimators": 800,  "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.0, "reg_lambda": 1.0},
#     {"num_leaves": 31, "min_child_samples": 20, "learning_rate": 0.02, "n_estimators": 1400, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.3, "reg_lambda": 1.0},
#     {"num_leaves": 31, "min_child_samples": 40, "learning_rate": 0.02, "n_estimators": 2200, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 1.5},
#     {"num_leaves": 63, "min_child_samples": 20, "learning_rate": 0.01, "n_estimators": 2200, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.3, "reg_lambda": 2.0},
#     {"num_leaves": 15, "min_child_samples": 80, "learning_rate": 0.02, "n_estimators": 1400, "subsample": 0.9, "colsample_bytree": 0.7, "reg_alpha": 0.7, "reg_lambda": 2.0},
#     {"num_leaves": 31, "min_child_samples": 80, "learning_rate": 0.01, "n_estimators": 2200, "subsample": 0.7, "colsample_bytree": 0.9, "reg_alpha": 0.3, "reg_lambda": 1.0},
#     {"num_leaves": 63, "min_child_samples": 40, "learning_rate": 0.02, "n_estimators": 1400, "subsample": 0.9, "colsample_bytree": 0.9, "reg_alpha": 0.0, "reg_lambda": 0.5},
#     {"num_leaves": 31, "min_child_samples": 20, "learning_rate": 0.01, "n_estimators": 3000, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.5, "reg_lambda": 2.0},
# ]
# results = []
# for i, params in enumerate(candidate_params, 1):
#     model = LGBMRegressor(objective="mae", random_state=42, n_jobs=-1, verbose=-1, **params)
#     model.fit(X_train_top, np.log1p(y_train))
#     pred = np.clip(np.expm1(model.predict(X_valid_top)), 0, None)
#     mae = mean_absolute_error(y_valid, pred)
#     sp  = spearmanr(y_valid, pred).statistic
#     results.append({"trial": i, "mae": mae, "spearman": sp, **params})
#     print(f"Trial {i}: MAE={mae:.4f}, Spearman={sp:.4f}")

# ── BEST MODEL — Trial 3 parameters ──────────────────────────────────────────
reg_final = LGBMRegressor(
    objective="mae",
    num_leaves=31,
    min_child_samples=20,
    learning_rate=0.005,
    n_estimators=5000,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
reg_final.fit(X_train_top, np.log1p(y_train))

y_pred_final = np.expm1(reg_final.predict(X_valid_top))
y_pred_final = np.clip(y_pred_final, 0, None)

print(f"MAE (BEST MODEL):      {mean_absolute_error(y_valid, y_pred_final):.2f}")
print(f"Spearman (BEST MODEL): {spearmanr(y_valid, y_pred_final).statistic:.3f}")

MAE (BEST MODEL):      63.13
Spearman (BEST MODEL): 0.392


## 7. Alternative Algorithms
We tested several other algorithms with the direct regression approach.
All performed similarly (~63-66 MAE), confirming LGBM MAE is the best choice.

In [19]:
# ── XGBoost with MAE objective ───────────────────────────────────────────────
# Best params from grid search: learning_rate=0.01, max_depth=6, n_estimators=700, subsample=0.7
reg_xgb_mae = XGBRegressor(
    objective="reg:absoluteerror",
    learning_rate=0.01, max_depth=6, n_estimators=700,
    subsample=0.7, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
reg_xgb_mae.fit(X_train, np.log1p(y_train))
y_pred_xgb_mae = np.clip(np.expm1(reg_xgb_mae.predict(X_valid)), 0, None)
print(f"MAE (XGBoost MAE):      {mean_absolute_error(y_valid, y_pred_xgb_mae):.2f}")
print(f"Spearman (XGBoost MAE): {spearmanr(y_valid, y_pred_xgb_mae).statistic:.3f}")

# ── Ensemble LGBM + XGBoost (did not improve) ────────────────────────────────
y_pred_ensemble = (y_pred_lgbm_best + y_pred_xgb_mae) / 2
print(f"\nMAE (Ensemble 50/50):      {mean_absolute_error(y_valid, y_pred_ensemble):.2f}")
print(f"Spearman (Ensemble 50/50): {spearmanr(y_valid, y_pred_ensemble).statistic:.3f}")
print("→ Ensemble did not improve over LGBM alone")

# ── Ridge / Lasso (did not improve) ──────────────────────────────────────────
# scaler_reg = StandardScaler()
# ridge = Ridge(alpha=10.0)
# ridge.fit(scaler_reg.fit_transform(X_train), np.log1p(y_train))
# MAE ~67 — linear models capture less complexity than tree-based models

# ── XGBoost Tweedie (did not improve) ────────────────────────────────────────
# reg_tweedie = XGBRegressor(objective="reg:tweedie", tweedie_variance_power=1.5, ...)
# MAE 77.05 — Tweedie worse despite being theoretically appropriate for zero-inflated data

# ── Random Forest direct (did not improve) ────────────────────────────────────
# reg_rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=5, ...)
# MAE 66.25 — similar to XGBoost MSE

MAE (XGBoost MAE):      63.17
Spearman (XGBoost MAE): 0.388

MAE (Ensemble 50/50):      63.10
Spearman (Ensemble 50/50): 0.385
→ Ensemble did not improve over LGBM alone


## 8. Automated Feature Engineering (FeatureTools)
We tested FeatureTools Deep Feature Synthesis (DFS) to automatically generate features.  
**Result:** No improvement. Manual domain-specific features outperformed automated ones.

| Approach | MAE | Spearman |
|---|---|---|
| FeatureTools only | 72.30 | 0.389 |
| Manual + FeatureTools | 71.25 | 0.402 |
| Manual only | 65.87 | 0.391 |

**Conclusion:** DFS generates generic aggregations. Our manual features capture domain knowledge
(return behavior, trend, VIP status) that DFS misses. More features ≠ better model.

In [20]:
# FeatureTools code (commented out — takes ~5 min to run):
#
# import featuretools as ft
# es = ft.EntitySet(id="clv_data")
# es = es.add_dataframe(dataframe_name="transactions", dataframe=df_train,
#                       index="index", make_index=True)
# es = es.add_dataframe(dataframe_name="customers",
#                       dataframe=df_train[["cust_id"]].drop_duplicates(), index="cust_id")
# es = es.add_relationship("customers", "cust_id", "transactions", "cust_id")
#
# feature_matrix, feature_defs = ft.dfs(
#     entityset=es, target_dataframe_name="customers",
#     agg_primitives=["sum","mean","max","min","std","count","percent_true","num_unique","last"],
#     trans_primitives=[], max_depth=1, verbose=True
# )
# # Generated 78 features in ~47 seconds
#
# # For valid: use calculate_feature_matrix (NOT dfs — avoids leakage)
# feature_matrix_valid = ft.calculate_feature_matrix(
#     features=feature_defs, entityset=es_valid, verbose=True
# )
print("FeatureTools experiment documented above — see comments")

FeatureTools experiment documented above — see comments


## 9. Probabilistic Models — BG/NBD + Gamma-Gamma
The BG/NBD model is the industry standard for CLV prediction. It models:
- **BG/NBD:** probability that a customer is still "alive" and expected future purchases
- **Gamma-Gamma:** expected monetary value per transaction

**Result:** MAE 72.53 — worse than our ML approach. Using BG/NBD predictions as a feature
for XGBoost also did not improve (MAE 65.87 — same as without it). The ML model already
captures the same information through frequency, recency, and T implicitly.

In [21]:
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

# ── Build BG/NBD input tables ─────────────────────────────────────────────────
bgf_train = summary_data_from_transaction_data(
    df_train, customer_id_col="cust_id", datetime_col="order_date",
    observation_period_end=pd.Timestamp("2017-12-31"), freq="D"
)
bgf_valid = summary_data_from_transaction_data(
    df_valid, customer_id_col="cust_id", datetime_col="order_date",
    observation_period_end=pd.Timestamp("2017-12-31"), freq="D"
)

# ── Stage 1: BG/NBD ──────────────────────────────────────────────────────────
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(bgf_train["frequency"], bgf_train["recency"], bgf_train["T"])
print(bgf)

# ── Stage 2: Gamma-Gamma ─────────────────────────────────────────────────────
returning_customers = bgf_train[bgf_train["frequency"] > 0].copy()
monetary = df_train_final.set_index("cust_id")["avg_ticket"]
returning_customers = returning_customers.join(monetary, how="left")
returning_customers = returning_customers[returning_customers["avg_ticket"] > 0].dropna(subset=["avg_ticket"])

ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(returning_customers["frequency"], returning_customers["avg_ticket"])
print(ggf)

<lifetimes.BetaGeoFitter: fitted with 93272 subjects, a: 0.10, alpha: 217.33, b: 0.36, r: 0.45>
<lifetimes.GammaGammaFitter: fitted with 30891 subjects, p: 4.38, q: 0.62, v: 4.16>


In [22]:
# ── Evaluate BG/NBD + Gamma-Gamma pipeline ───────────────────────────────────
valid_returning = bgf_valid[bgf_valid["frequency"] > 0].copy()
monetary_valid  = df_valid_final.set_index("cust_id")["avg_ticket"]
valid_returning = valid_returning.join(monetary_valid, how="left")
valid_returning = valid_returning[valid_returning["avg_ticket"] > 0].dropna(subset=["avg_ticket"])

valid_returning["predicted_clv"] = ggf.customer_lifetime_value(
    bgf, valid_returning["frequency"], valid_returning["recency"],
    valid_returning["T"], valid_returning["avg_ticket"],
    time=24, freq="D", discount_rate=0.0
)

y_pred_bgf = bgf_valid[["frequency"]].copy()
y_pred_bgf = y_pred_bgf.join(valid_returning["predicted_clv"], how="left")
y_pred_bgf["predicted_clv"] = y_pred_bgf["predicted_clv"].fillna(0)
y_pred_bgf = y_pred_bgf.reindex(df_valid_final["cust_id"].values)
y_pred_survival = y_pred_bgf["predicted_clv"].values

print(f"MAE (BG/NBD + Gamma-Gamma): {mean_absolute_error(y_valid, y_pred_survival):.2f}")
print(f"Spearman:                   {spearmanr(y_valid, y_pred_survival).statistic:.3f}")
print("→ Worse than ML approach. Probabilistic models better suited for longer time horizons.")

MAE (BG/NBD + Gamma-Gamma): 72.53
Spearman:                   0.386
→ Worse than ML approach. Probabilistic models better suited for longer time horizons.


## 10. Error Analysis
We analyzed the biggest prediction errors to understand where the model fails.
This analysis led to the `is_vip` feature.

In [23]:
# ── Error analysis on best XGBoost direct model ──────────────────────────────
error_analysis = df_valid_final[["cust_id", "revenue_2018_2019"]].copy()
error_analysis["predicted"]    = y_pred_direct
error_analysis["error"]        = np.abs(error_analysis["revenue_2018_2019"] - error_analysis["predicted"])
error_analysis["error_signed"] = error_analysis["revenue_2018_2019"] - error_analysis["predicted"]
error_analysis = error_analysis.sort_values("error", ascending=False)

print("=== Top 10 biggest errors ===")
print(error_analysis.head(10).to_string(index=False))

print(f"\n=== Error distribution ===")
print(f"Customers with error > 500: {(error_analysis['error'] > 500).sum()}")
print(f"Customers with error > 200: {(error_analysis['error'] > 200).sum()}")
print(f"Customers with error < 50:  {(error_analysis['error'] < 50).sum()}")

print(f"\nUnderpredicting: {(error_analysis['error_signed'] > 0).sum()}")
print(f"Overpredicting:  {(error_analysis['error_signed'] < 0).sum()}")

=== Top 10 biggest errors ===
         cust_id  revenue_2018_2019  predicted       error  error_signed
vkkljb27ovnlzde6            1185.98   3.308792 1182.671208   1182.671208
xonhxxf3ijmyvqdv            1175.05   2.761661 1172.288339   1172.288339
mn5oibpoumhj4zml            1138.07   6.285840 1131.784160   1131.784160
7pmkwtfvwhtfzb3c            1168.70  42.370796 1126.329204   1126.329204
56fmop2mqsio4jrf            1109.65   6.301891 1103.348109   1103.348109
v2x73lef5pqu46zz            1103.16   3.741300 1099.418700   1099.418700
ar7fp4pg57rcw3jw            1105.34   6.718167 1098.621833   1098.621833
byu3nvqddsbw4nuw            1097.90   7.678872 1090.221128   1090.221128
q5byopumbg7szitu            1092.09   6.635345 1085.454655   1085.454655
uq62swkbvjinp7s3            1089.67   4.573219 1085.096781   1085.096781

=== Error distribution ===
Customers with error > 500: 465
Customers with error > 200: 2378
Customers with error < 50:  16159

Underpredicting: 8296
Overpredicting:  

In [24]:
# ── Profile of high-error customers ──────────────────────────────────────────
# Finding: 471 customers with error > 500 are high-value customers the model misses
# They have 2x frequency, 2x revenue, and bought more recently
# → Led to the is_vip feature (frequency >= 4 AND total_revenue >= 249)

high_error = error_analysis[error_analysis["error"] > 500].copy()
high_error_features = high_error.merge(df_valid_final, on="cust_id")
rest = df_valid_final[~df_valid_final["cust_id"].isin(high_error["cust_id"])]

print(f"High-error customers: {len(high_error_features)}")
print(f"\nFeature comparison — high error vs rest:")
for col in ["frequency", "total_revenue", "avg_ticket", "return_rate", "recency_days", "pct_orders_2017"]:
    print(f"  {col:30s} high_error: {high_error_features[col].mean():.2f} | rest: {rest[col].mean():.2f}")

High-error customers: 465

Feature comparison — high error vs rest:
  frequency                      high_error: 2.99 | rest: 1.49
  total_revenue                  high_error: 295.53 | rest: 122.49
  avg_ticket                     high_error: 103.35 | rest: 80.65
  return_rate                    high_error: 0.18 | rest: 0.09
  recency_days                   high_error: 183.42 | rest: 300.70
  pct_orders_2017                high_error: 0.68 | rest: 0.58


## 11. Clustering Experiments
We tried segmenting customers and training specialized models per segment.  
**Result:** No improvement over the global model in any configuration.  
LightGBM already learns customer segments internally — explicit clustering adds no value.

However, the clustering provides useful **business insights** about customer segments.

In [25]:
# ── K-Means k=5 clustering ───────────────────────────────────────────────────
cluster_cols = ["recency_days", "frequency", "total_revenue",
                "return_rate", "avg_ticket", "pct_orders_2017",
                "n_genders", "avg_days_between_orders"]

X_cluster       = df_train_final[cluster_cols].fillna(0)
X_cluster_valid = df_valid_final[cluster_cols].fillna(0)

scaler_km = StandardScaler()
X_cluster_scaled       = scaler_km.fit_transform(X_cluster)
X_cluster_valid_scaled = scaler_km.transform(X_cluster_valid)

# Elbow method showed k=5 optimal (commented out):
# for k in range(2,8): print(f"k={k}, inertia={KMeans(n_clusters=k).fit(X_cluster_scaled).inertia_:.0f}")

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_train_final["cluster"] = kmeans.fit_predict(X_cluster_scaled)
df_valid_final["cluster"] = kmeans.predict(X_cluster_valid_scaled)

print("=== Cluster profiles ===")
print(df_train_final.groupby("cluster")[["recency_days","frequency","total_revenue","revenue_2018_2019"]].mean().round(2))
print("\nCluster sizes:")
print(df_train_final["cluster"].value_counts().sort_index())

=== Cluster profiles ===
         recency_days  frequency  total_revenue  revenue_2018_2019
cluster                                                           
0              153.42       2.02         166.25             104.22
1              550.00       1.13          89.66              32.96
2              187.33       1.17          90.61              50.65
3              259.88       1.43         105.61              83.21
4              150.65       4.27         401.44             241.42

Cluster sizes:
cluster
0     6203
1    27230
2    39693
3    11837
4     8309
Name: count, dtype: int64


In [26]:
# ── Best model per cluster (heterogeneous ensemble) ──────────────────────────
# We tested LGBM MAE, XGBoost MAE, RF, and Ridge per cluster
# XGBoost MAE won on most clusters, LGBM MAE won on cluster 4
# Combined MAE: 63.31 — worse than global model (63.05)
#
# Key finding from cluster analysis:
# Cluster VIP (high frequency/revenue) has MAE ~177 — these customers drive most of the error
# No single algorithm solves this — it's a fundamental data limitation

models_to_try = {
    "LGBM_MAE": LGBMRegressor(objective="mae", num_leaves=31, min_child_samples=20,
                               learning_rate=0.005, n_estimators=5000, subsample=0.8,
                               colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
                               random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost_MAE": XGBRegressor(objective="reg:absoluteerror", n_estimators=700,
                                 max_depth=6, learning_rate=0.01, subsample=0.7,
                                 colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0),
}

best_models = {}
for cluster_id in range(5):
    train_mask = df_train_final["cluster"] == cluster_id
    valid_mask = df_valid_final["cluster"] == cluster_id
    X_c_train  = df_train_final[train_mask][top_features]
    y_c_train  = np.log1p(df_train_final[train_mask]["revenue_2018_2019"])
    X_c_valid  = df_valid_final[valid_mask][top_features]
    y_c_valid  = df_valid_final[valid_mask]["revenue_2018_2019"]

    best_mae, best_name, best_model = np.inf, None, None
    for name, model in models_to_try.items():
        m = copy.deepcopy(model)
        m.fit(X_c_train, y_c_train)
        pred = np.clip(np.expm1(m.predict(X_c_valid)), 0, None)
        mae  = mean_absolute_error(y_c_valid, pred)
        if mae < best_mae:
            best_mae, best_name, best_model = mae, name, m
    best_models[cluster_id] = (best_name, best_model)
    print(f"Cluster {cluster_id}: best={best_name} MAE={best_mae:.2f}")

y_pred_cluster = np.zeros(len(df_valid_final))
for cluster_id, (name, model) in best_models.items():
    mask = df_valid_final["cluster"] == cluster_id
    y_pred_cluster[mask] = np.clip(np.expm1(model.predict(df_valid_final[mask][top_features])), 0, None)

print(f"\nMAE (best per cluster): {mean_absolute_error(y_valid, y_pred_cluster):.2f}")
print("→ No improvement over global model")

Cluster 0: best=XGBoost_MAE MAE=93.53
Cluster 1: best=XGBoost_MAE MAE=31.59
Cluster 2: best=XGBoost_MAE MAE=49.99
Cluster 3: best=LGBM_MAE MAE=81.71
Cluster 4: best=XGBoost_MAE MAE=178.02

MAE (best per cluster): 63.31
→ No improvement over global model


## 12. Final Model & Results Summary

| Model | MAE | Spearman | Notes |
|---|---|---|---|
| RF two-stage (baseline) | ~81 | 0.396 | First attempt |
| XGBoost two-stage + log | 71.16 | 0.399 | Log transform helped |
| XGBoost direct + log | 66.24 | 0.392 | Key insight: skip classifier |
| XGBoost direct + grid search | 66.01 | 0.392 | Minor improvement |
| LGBM objective=MAE | 63.44 | 0.393 | **Big jump: optimize right metric** |
| LGBM MAE + grid search | 63.36 | 0.391 | Small improvement |
| **LGBM MAE + num_leaves tuning** | **63.05** | **0.391** | **Best model** |
| Clustering (various) | 63.31-66.10 | ~0.38 | No improvement |
| BG/NBD + Gamma-Gamma | 72.53 | 0.386 | Probabilistic model worse |
| **Benchmark to beat** | **61.15** | **0.41** | |

**Best model: LGBM MAE with log1p target, top 24 features by importance**  
**Key discoveries:**
1. Direct regression > two-stage pipeline
2. Optimizing MAE directly gave the biggest single improvement (~3 points)
3. Feature engineering matters more than algorithm choice
4. High-value customers (VIP cluster) drive most of the remaining error

In [27]:
# ── Final model evaluation ────────────────────────────────────────────────────
print("=" * 50)
print("FINAL MODEL RESULTS")
print("=" * 50)
print(f"MAE:      {mean_absolute_error(y_valid, y_pred_final):.2f}  (benchmark: 61.15)")
print(f"Spearman: {spearmanr(y_valid, y_pred_final).statistic:.3f} (benchmark: 0.41)")
print()
print("Model: LGBMRegressor — direct regression, objective=mae, log1p target")
print(f"Features: top {len(top_features)} by importance")
print("Params: num_leaves=31, min_child_samples=20, learning_rate=0.005,")
print("        n_estimators=5000, subsample=0.8, colsample_bytree=0.8,")
print("        reg_alpha=0.5, reg_lambda=2.0")

FINAL MODEL RESULTS
MAE:      63.13  (benchmark: 61.15)
Spearman: 0.392 (benchmark: 0.41)

Model: LGBMRegressor — direct regression, objective=mae, log1p target
Features: top 23 by importance
Params: num_leaves=31, min_child_samples=20, learning_rate=0.005,
        n_estimators=5000, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.5, reg_lambda=2.0


In [29]:
import featuretools as ft
import warnings
warnings.filterwarnings("ignore")

# Get train customers for VIP cluster only
vip_cluster_id = 1  # cluster with frequency 4.27, revenue 401
vip_train_custs = df_train_final[df_train_final["cluster"] == vip_cluster_id]["cust_id"].astype(str).tolist()
vip_valid_custs = df_valid_final[df_valid_final["cluster"] == vip_cluster_id]["cust_id"].astype(str).tolist()

df_train_vip = df_train[df_train["cust_id"].astype(str).isin(vip_train_custs)].copy()
df_valid_vip = df_valid[df_valid["cust_id"].astype(str).isin(vip_valid_custs)].copy()

print(f"VIP train transactions: {len(df_train_vip)}")
print(f"VIP valid transactions: {len(df_valid_vip)}")
print(f"VIP train customers: {df_train_vip['cust_id'].nunique()}")

# Build EntitySet for VIP cluster
es_vip = ft.EntitySet(id="vip_data")
es_vip = es_vip.add_dataframe(
    dataframe_name="transactions", dataframe=df_train_vip,
    index="index", make_index=True
)
es_vip = es_vip.add_dataframe(
    dataframe_name="customers",
    dataframe=df_train_vip[["cust_id"]].drop_duplicates(),
    index="cust_id"
)
es_vip = es_vip.add_relationship("customers", "cust_id", "transactions", "cust_id")

feature_matrix_vip, feature_defs_vip = ft.dfs(
    entityset=es_vip, target_dataframe_name="customers",
    agg_primitives=["sum","mean","max","min","std","count","percent_true","num_unique","last"],
    trans_primitives=[], max_depth=1, verbose=True
)

print(f"\nFeatures generated: {feature_matrix_vip.shape[1]}")

C:\Users\flopa\anaconda3\Lib\site-packages\woodwork\__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


VIP train transactions: 37771
VIP valid transactions: 9523
VIP train customers: 27230
Built 72 features
Elapsed: 00:17 | Progress: 100%|███████████████████████████████████████████████████████████████████████████████████████

Features generated: 72


In [30]:
# ── Clean FeatureTools features for VIP cluster ───────────────────────────────
feature_matrix_vip = feature_matrix_vip.reset_index()

# Drop LAST columns (strings)
last_cols = [c for c in feature_matrix_vip.columns if c.startswith("LAST(")]
feature_matrix_vip = feature_matrix_vip.drop(columns=last_cols)

# Fill NaNs
std_cols = [c for c in feature_matrix_vip.columns if c.startswith("STD(")]
feature_matrix_vip[std_cols] = feature_matrix_vip[std_cols].fillna(0)
feature_matrix_vip = feature_matrix_vip.fillna(0)

# Merge with targets and manual features
df_vip_train_ft = df_train_final[df_train_final["cluster"] == vip_cluster_id].merge(
    feature_matrix_vip, on="cust_id", how="left"
)

# Build valid FeatureTools features
es_vip_valid = ft.EntitySet(id="vip_valid")
es_vip_valid = es_vip_valid.add_dataframe(
    dataframe_name="transactions", dataframe=df_valid_vip,
    index="index", make_index=True
)
es_vip_valid = es_vip_valid.add_dataframe(
    dataframe_name="customers",
    dataframe=df_valid_vip[["cust_id"]].drop_duplicates(),
    index="cust_id"
)
es_vip_valid = es_vip_valid.add_relationship("customers", "cust_id", "transactions", "cust_id")

feature_matrix_vip_valid = ft.calculate_feature_matrix(
    features=feature_defs_vip, entityset=es_vip_valid, verbose=True
).reset_index()

last_cols_v = [c for c in feature_matrix_vip_valid.columns if c.startswith("LAST(")]
feature_matrix_vip_valid = feature_matrix_vip_valid.drop(columns=last_cols_v)
feature_matrix_vip_valid[std_cols] = feature_matrix_vip_valid[std_cols].fillna(0)
feature_matrix_vip_valid = feature_matrix_vip_valid.fillna(0)

df_vip_valid_ft = df_valid_final[df_valid_final["cluster"] == vip_cluster_id].merge(
    feature_matrix_vip_valid, on="cust_id", how="left"
)

# Feature cols: manual + featuretools
ft_cols = [c for c in feature_matrix_vip.columns if c != "cust_id"]
feature_cols_vip = top_features + [c for c in ft_cols if c not in top_features]

X_vip_train = df_vip_train_ft[feature_cols_vip].fillna(0)
y_vip_train = np.log1p(df_vip_train_ft["revenue_2018_2019"])
X_vip_valid = df_vip_valid_ft[feature_cols_vip].fillna(0)
y_vip_valid = df_vip_valid_ft["revenue_2018_2019"]

# Train model
reg_vip_ft = LGBMRegressor(
    objective="mae", num_leaves=31, min_child_samples=20,
    learning_rate=0.005, n_estimators=5000, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
    random_state=42, n_jobs=-1, verbose=-1
)
reg_vip_ft.fit(X_vip_train, y_vip_train)

pred_vip = np.clip(np.expm1(reg_vip_ft.predict(X_vip_valid)), 0, None)
mae_vip_ft = mean_absolute_error(y_vip_valid, pred_vip)
print(f"MAE VIP cluster (manual features):              177.53")
print(f"MAE VIP cluster (manual + FeatureTools):        {mae_vip_ft:.2f}")

Elapsed: 00:04 | Progress: 100%|███████████████████████████████████████████████████████████████████████████████████████
MAE VIP cluster (manual features):              177.53
MAE VIP cluster (manual + FeatureTools):        31.74


In [32]:
import featuretools as ft
import warnings
warnings.filterwarnings("ignore")

def clean_ft_features(fm):
    fm = fm.reset_index()
    # Drop LAST columns (strings)
    last_cols = [c for c in fm.columns if c.startswith("LAST(")]
    fm = fm.drop(columns=last_cols)
    # Convert categorical to numeric/string before fillna
    for col in fm.columns:
        if hasattr(fm[col], 'cat'):
            fm[col] = fm[col].astype(str)
    for col in fm.columns:
        if fm[col].dtype == object:
            fm[col] = fm[col].fillna("unknown")
        else:
            fm[col] = fm[col].fillna(0)
    return fm

# ── FeatureTools for ALL clusters ─────────────────────────────────────────────
cluster_ft_models = {}
y_pred_ft_clusters = np.zeros(len(df_valid_final))

for cluster_id in range(5):
    print(f"\n--- Cluster {cluster_id} ---")
    
    train_custs = df_train_final[df_train_final["cluster"] == cluster_id]["cust_id"].astype(str).tolist()
    valid_custs = df_valid_final[df_valid_final["cluster"] == cluster_id]["cust_id"].astype(str).tolist()
    
    df_tr = df_train[df_train["cust_id"].astype(str).isin(train_custs)].copy()
    df_va = df_valid[df_valid["cust_id"].astype(str).isin(valid_custs)].copy()
    
    # Build EntitySet for train
    es_tr = ft.EntitySet(id=f"cluster_{cluster_id}_train")
    es_tr = es_tr.add_dataframe(dataframe_name="transactions", dataframe=df_tr,
                                 index="index", make_index=True)
    es_tr = es_tr.add_dataframe(dataframe_name="customers",
                                 dataframe=df_tr[["cust_id"]].drop_duplicates(), index="cust_id")
    es_tr = es_tr.add_relationship("customers", "cust_id", "transactions", "cust_id")
    
    fm_tr, fd = ft.dfs(entityset=es_tr, target_dataframe_name="customers",
                       agg_primitives=["sum","mean","max","min","std","count",
                                       "percent_true","num_unique","last"],
                       trans_primitives=[], max_depth=1, verbose=False)
    fm_tr = clean_ft_features(fm_tr)
    
    # Build EntitySet for valid
    es_va = ft.EntitySet(id=f"cluster_{cluster_id}_valid")
    es_va = es_va.add_dataframe(dataframe_name="transactions", dataframe=df_va,
                                 index="index", make_index=True)
    es_va = es_va.add_dataframe(dataframe_name="customers",
                                 dataframe=df_va[["cust_id"]].drop_duplicates(), index="cust_id")
    es_va = es_va.add_relationship("customers", "cust_id", "transactions", "cust_id")
    
    fm_va = ft.calculate_feature_matrix(features=fd, entityset=es_va, verbose=False)
    fm_va = clean_ft_features(fm_va)
    
    # Merge with manual features
    df_tr_final = df_train_final[df_train_final["cluster"] == cluster_id].merge(fm_tr, on="cust_id", how="left")
    df_va_final = df_valid_final[df_valid_final["cluster"] == cluster_id].merge(fm_va, on="cust_id", how="left")
    
    ft_only   = [c for c in fm_tr.columns if c != "cust_id" and fm_tr[c].dtype != object]
    feat_cols = top_features + [c for c in ft_only if c not in top_features]
    
    X_tr = df_tr_final[feat_cols].fillna(0)
    y_tr = np.log1p(df_tr_final["revenue_2018_2019"])
    X_va = df_va_final[feat_cols].fillna(0)
    y_va = df_va_final["revenue_2018_2019"]
    
    model = LGBMRegressor(
        objective="mae", num_leaves=31, min_child_samples=20,
        learning_rate=0.005, n_estimators=5000, subsample=0.8,
        colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, n_jobs=-1, verbose=-1
    )
    model.fit(X_tr, y_tr)
    cluster_ft_models[cluster_id] = (model, feat_cols, fd)
    
    pred = np.clip(np.expm1(model.predict(X_va)), 0, None)
    mae  = mean_absolute_error(y_va, pred)
    
    valid_idx = df_valid_final[df_valid_final["cluster"] == cluster_id].index
    y_pred_ft_clusters[valid_idx] = pred
    print(f"  MAE: {mae:.2f}")

mae_ft_all      = mean_absolute_error(y_valid, y_pred_ft_clusters)
spearman_ft_all = spearmanr(y_valid, y_pred_ft_clusters).statistic
print(f"\nMAE (FeatureTools per cluster): {mae_ft_all:.2f}")
print(f"Spearman:                       {spearman_ft_all:.3f}")



--- Cluster 0 ---
  MAE: 94.49

--- Cluster 1 ---
  MAE: 31.74

--- Cluster 2 ---
  MAE: 50.23

--- Cluster 3 ---
  MAE: 82.04

--- Cluster 4 ---
  MAE: 179.90

MAE (FeatureTools per cluster): 63.73
Spearman:                       0.387


In [33]:
print("=== Cluster profiles actuales ===")
print(df_train_final.groupby("cluster")[
    ["recency_days", "frequency", "total_revenue", "revenue_2018_2019"]
].mean().round(2))

=== Cluster profiles actuales ===
         recency_days  frequency  total_revenue  revenue_2018_2019
cluster                                                           
0              153.42       2.02         166.25             104.22
1              550.00       1.13          89.66              32.96
2              187.33       1.17          90.61              50.65
3              259.88       1.43         105.61              83.21
4              150.65       4.27         401.44             241.42


In [34]:
# ── FeatureTools specifically for VIP cluster (4) with more primitives ────────
vip_id = 4
vip_train_custs = df_train_final[df_train_final["cluster"] == vip_id]["cust_id"].astype(str).tolist()
vip_valid_custs = df_valid_final[df_valid_final["cluster"] == vip_id]["cust_id"].astype(str).tolist()

df_tr_vip = df_train[df_train["cust_id"].astype(str).isin(vip_train_custs)].copy()
df_va_vip = df_valid[df_valid["cust_id"].astype(str).isin(vip_valid_custs)].copy()

es_vip4 = ft.EntitySet(id="vip4_train")
es_vip4 = es_vip4.add_dataframe(dataframe_name="transactions", dataframe=df_tr_vip,
                                  index="index", make_index=True)
es_vip4 = es_vip4.add_dataframe(dataframe_name="customers",
                                  dataframe=df_tr_vip[["cust_id"]].drop_duplicates(), index="cust_id")
es_vip4 = es_vip4.add_relationship("customers", "cust_id", "transactions", "cust_id")

fm_vip4, fd_vip4 = ft.dfs(
    entityset=es_vip4, target_dataframe_name="customers",
    agg_primitives=["sum","mean","max","min","std","count","percent_true",
                    "num_unique","last","trend","skew","median"],
    trans_primitives=[], max_depth=1, verbose=True
)
fm_vip4 = clean_ft_features(fm_vip4)

# Valid
es_vip4_valid = ft.EntitySet(id="vip4_valid")
es_vip4_valid = es_vip4_valid.add_dataframe(dataframe_name="transactions", dataframe=df_va_vip,
                                              index="index", make_index=True)
es_vip4_valid = es_vip4_valid.add_dataframe(dataframe_name="customers",
                                              dataframe=df_va_vip[["cust_id"]].drop_duplicates(), index="cust_id")
es_vip4_valid = es_vip4_valid.add_relationship("customers", "cust_id", "transactions", "cust_id")

fm_vip4_valid = ft.calculate_feature_matrix(features=fd_vip4, entityset=es_vip4_valid, verbose=False)
fm_vip4_valid = clean_ft_features(fm_vip4_valid)

# Merge and train
df_tr_vip4 = df_train_final[df_train_final["cluster"] == vip_id].merge(fm_vip4, on="cust_id", how="left")
df_va_vip4 = df_valid_final[df_valid_final["cluster"] == vip_id].merge(fm_vip4_valid, on="cust_id", how="left")

ft_only_vip4 = [c for c in fm_vip4.columns if c != "cust_id" and fm_vip4[c].dtype != object]
feat_cols_vip4 = top_features + [c for c in ft_only_vip4 if c not in top_features]

X_tr_vip4 = df_tr_vip4[feat_cols_vip4].fillna(0)
y_tr_vip4 = np.log1p(df_tr_vip4["revenue_2018_2019"])
X_va_vip4 = df_va_vip4[feat_cols_vip4].fillna(0)
y_va_vip4 = df_va_vip4["revenue_2018_2019"]

reg_vip4 = LGBMRegressor(
    objective="mae", num_leaves=31, min_child_samples=20,
    learning_rate=0.005, n_estimators=5000, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
    random_state=42, n_jobs=-1, verbose=-1
)
reg_vip4.fit(X_tr_vip4, y_tr_vip4)

pred_vip4 = np.clip(np.expm1(reg_vip4.predict(X_va_vip4)), 0, None)
print(f"MAE VIP cluster sin FeatureTools: 179.90")
print(f"MAE VIP cluster con FeatureTools: {mean_absolute_error(y_va_vip4, pred_vip4):.2f}")

Built 86 features
Elapsed: 00:13 | Progress: 100%|███████████████████████████████████████████████████████████████████████████████████████
MAE VIP cluster sin FeatureTools: 179.90
MAE VIP cluster con FeatureTools: 179.64


In [35]:
# Try sqrt transform for VIP cluster
reg_vip4_sqrt = LGBMRegressor(
    objective="mae", num_leaves=31, min_child_samples=20,
    learning_rate=0.005, n_estimators=5000, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
    random_state=42, n_jobs=-1, verbose=-1
)
reg_vip4_sqrt.fit(X_tr_vip4, np.sqrt(y_tr_vip4.apply(lambda x: np.expm1(x))))

pred_vip4_sqrt = reg_vip4_sqrt.predict(X_va_vip4) ** 2
pred_vip4_sqrt = np.clip(pred_vip4_sqrt, 0, None)

print(f"MAE VIP (log transform):  179.64")
print(f"MAE VIP (sqrt transform): {mean_absolute_error(y_va_vip4, pred_vip4_sqrt):.2f}")

MAE VIP (log transform):  179.64
MAE VIP (sqrt transform): 177.14


In [36]:
# ── Best transform per cluster ────────────────────────────────────────────────
from sklearn.preprocessing import PowerTransformer

transforms = {
    "log1p": (np.log1p, np.expm1),
    "sqrt":  (np.sqrt,  lambda x: x**2),
    "none":  (lambda x: x, lambda x: x),
}

best_transform_models = {}
y_pred_best_transform = np.zeros(len(df_valid_final))

for cluster_id in range(5):
    train_mask = df_train_final["cluster"] == cluster_id
    valid_mask = df_valid_final["cluster"] == cluster_id

    X_tr = df_train_final[train_mask][top_features]
    y_tr = df_train_final[train_mask]["revenue_2018_2019"]
    X_va = df_valid_final[valid_mask][top_features]
    y_va = df_valid_final[valid_mask]["revenue_2018_2019"]

    print(f"\n--- Cluster {cluster_id} ---")
    best_mae, best_name, best_model = np.inf, None, None

    for name, (transform_fn, inverse_fn) in transforms.items():
        m = LGBMRegressor(
            objective="mae", num_leaves=31, min_child_samples=20,
            learning_rate=0.005, n_estimators=5000, subsample=0.8,
            colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
            random_state=42, n_jobs=-1, verbose=-1
        )
        m.fit(X_tr, transform_fn(y_tr))
        pred = np.clip(inverse_fn(m.predict(X_va)), 0, None)
        mae  = mean_absolute_error(y_va, pred)
        print(f"  {name:8s} MAE: {mae:.2f}")

        if mae < best_mae:
            best_mae, best_name, best_model = mae, name, (m, inverse_fn)

    best_transform_models[cluster_id] = best_model
    print(f"  → Best: {best_name} ({best_mae:.2f})")

    valid_idx = df_valid_final[valid_mask].index
    m, inv = best_transform_models[cluster_id]
    y_pred_best_transform[valid_idx] = np.clip(inv(m.predict(X_va)), 0, None)

mae_bt      = mean_absolute_error(y_valid, y_pred_best_transform)
spearman_bt = spearmanr(y_valid, y_pred_best_transform).statistic
print(f"\nMAE (best transform per cluster): {mae_bt:.2f}")
print(f"Spearman:                         {spearman_bt:.3f}")


--- Cluster 0 ---
  log1p    MAE: 95.97
  sqrt     MAE: 95.78
  none     MAE: 96.17
  → Best: sqrt (95.78)

--- Cluster 1 ---
  log1p    MAE: 31.72
  sqrt     MAE: 31.72
  none     MAE: 31.98
  → Best: sqrt (31.72)

--- Cluster 2 ---
  log1p    MAE: 50.34
  sqrt     MAE: 50.14
  none     MAE: 50.37
  → Best: sqrt (50.14)

--- Cluster 3 ---
  log1p    MAE: 81.71
  sqrt     MAE: 81.60
  none     MAE: 81.63
  → Best: sqrt (81.60)

--- Cluster 4 ---
  log1p    MAE: 180.11
  sqrt     MAE: 178.01
  none     MAE: 177.14
  → Best: none (177.14)

MAE (best transform per cluster): 63.46
Spearman:                         0.375
